# Chapter 4 · Wall Torques for Active Brownian Particles — Bayesian Analysis

Chapter 4 (pp. 59–83; paper arXiv:2409.03533), Figures 21, 26, 27, 29, 30 and 31.

| Figure | Page | Content |
|---|---|---|
| Figure 21 | p. 61 | Single rods colliding with a wall |
| Figure 22 | p. 63 | Simulation setup (illustration, not computed) |
| Figures 23, 24 | pp. 64, 66 | Least-squares analysis (by M. Duchêne, not part of this notebook) |
| Figure 25 | p. 70 | Inference process (illustration, not computed) |
| Figure 26 | p. 72 | Filtering of the orientation |
| Figure 27 | p. 74 | Kalman filter vs. particle filter |
| Figure 29 | p. 76 | Quadratic B-spline basis |
| Figure 30 | p. 78 | Test case on type-A data |
| Figure 31 | p. 79 | Torque function of active rods |

The posterior distributions (Figures 30C and 31) are sampled with the ensemble MCMC sampler `emcee`.

**Switch** (set in the cell before Figure 27): `FULL_RESOLUTION = True` uses 1500 filter particles and 3000 ensemble steps
of 32 walkers for the posterior sampling (hours); `False` uses 100 particles and 1000 steps of 16 walkers.

The likelihood grids of Figures 27 and 30A/B were computed on an HPC cluster (1000 × 1000 and 2000 × 2000 points, each
grid split into 40 × 40 chunks computed as separate jobs). Here they are evaluated on coarse grids with the same
filter settings. The job scripts and the merging of the chunks are in the branch `original_scripts` of this repository
(folders `fig27_kalman_vs_particle` and `fig30_typeA_test`).

In [ ]:
import time
from datetime import datetime, timedelta
import numpy as np
import matplotlib.pyplot as plt
import emcee

import lambert_thesis as lt
from lambert_thesis import walltorque as wt, plotting
from lambert_thesis.plotting import savefig

plotting.setup()
rng = lt.set_seed(lt.SEED)

## 4.2 Model

The base model is a 2D ABP with anisotropic translational diffusion,

$$\dot{\mathbf r}=v_0\hat\varphi\,\mathbf e_x+\frac1{k_BT}\hat\varphi^{-1}\mathbf D_T\hat\varphi\,\mathbf F_\text{wall}+\sqrt2\,\hat\varphi^{-1}\sqrt{\mathbf D_T}\,\boldsymbol\xi,\qquad
\dot\varphi=\frac{D_R}{k_BT}\tau_\text{ABP}+\sqrt{2D_R}\,\chi,$$

with diffusion constants of a spherocylinder (Lüders et al.). Walls repel with a WCA potential (cut-off $x_c=2^{1/6}\sigma/2$,
$\epsilon=4k_BT$), and the ABP is subject to an empirical torque $\tau_\text{ABP}=|\mathbf F_\text{wall}|\cdot f(\theta_\text{wall}-\varphi)$,
with the angle term expanded into modes $f(\varphi)=\sum_n\alpha_n\sin(n[\theta_\text{wall}-\varphi])$.

Two types of trajectories are generated: **type A** — ABPs with a predefined torque function (ground truth known) — and
**type R** — rods without empirical torque, whose WCA forces act on test sites along the rod and create torques through
their lever arms. The observed location of a rod is its tip.

*Code:* `walltorque.simulate_abp` and `walltorque.simulate_rod` (Euler–Maruyama, wall at $x=0$, particles start at negative $x$).

### Figure 21 (p. 61): single rods colliding with a wall
Rods with aspect ratio $p=5$, Péclet number 100, $\epsilon=4$, started at $(-5, 0)$ with orientations from $-60°$ to $60°$.

In [ ]:
phi0 = np.deg2rad(np.linspace(-60, 60, 25))
xr, yr, _ = wt.simulate_rod(3500, 1e-4, phi0, Pe=100, Dt=1.0, aspect_ratio=5, rng=rng, x0=-5.0, eps=4.0)

plt.figure(figsize=(10, 2.6))
plt.plot(yr.T, xr.T, lw=0.5)
plt.plot(yr[:, 0], xr[:, 0], "ro", ms=2, label="start")
plt.axhline(wt.WCA_CUT, color="g", ls="--", label="WCA cut-off")
plt.gca().invert_yaxis()
plt.xlabel("y"); plt.ylabel("x"); plt.grid(); plt.legend(loc="lower right", fontsize=7)
savefig("fig21_rod_collisions");

**Figure 21 (p. 61).** Trajectories of single rods colliding with a flat wall at $x=0$; the green dashed line is the WCA
cut-off. The traces are drawn using the rods' tips.

## 4.4 Bayesian analysis

We learn the torque function from positions only (no orientations). After an Euler–Maruyama step the deterministic
displacement for a given orientation is known, and the remaining offset $\mathbf o_i^{\varphi_i}$ is Gaussian with covariance
$2\hat\varphi_i\mathbf D_T\hat\varphi_i^{-1}\Delta t$. The unobserved orientations are marginalised with a **particle filter**:
particles carry orientations, are weighted by the offset likelihood at every step, resampled, and then propagated with the
torque and rotational diffusion,

$$p(Y\mid\theta)\approx\prod_i\frac1{N_p}\sum_k p(\mathbf o_i^{\tilde\varphi_i^k}\mid\theta).$$

*Code:* `walltorque.loglik_particle_filter(x, y, phi0, dt, Pe, Dt, Dr, eps, kind, q2, params, n_particles, seed)` — a numba
kernel, parallel over the parameter sets (rows of `params`); every parameter set uses its own seed, and with the same seed
the estimate is a deterministic function of the parameters (common random numbers).

**Type-A data** (single mode $f=\alpha\sin(-\varphi/k)$, $\alpha=10$, $k=1$): Péclet number 1, $D_T=D_R=10^{-3}$,
$\Delta t=0.01$, start at $x=-1.5$, initial orientations in $\pm45°$.

In [ ]:
A = dict(Pe=1.0, Dt=1e-3, Dr=1e-3, dt=0.01, eps=1.0, q2=wt.Q2)
GT_SINGLE = np.array([10.0, 1.0])

### Figure 26 (p. 72): filtering of the orientation
One trajectory of 1000 steps; the particle swarm at every step gives the posterior predictive of the orientation.

In [ ]:
x1, y1, p1 = wt.simulate_abp(1000, A["dt"], np.array([0.8]), A["Pe"], A["Dt"], A["Dr"], GT_SINGLE, rng, x0=-1.5,
                             kind="single_mode")

def filter_bands(params):
    _, swarm = wt.loglik_particle_filter(x1, y1, p1[:, 0], A["dt"], A["Pe"], A["Dt"], A["Dr"], A["eps"],
                                         wt.TORQUES["single_mode"], A["q2"], np.array([params]), 1500, lt.SEED,
                                         keep_phi=True)
    return np.percentile(swarm[0, :-1], [2.5, 16, 84, 97.5], axis=1)

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
for ax, params, title in [(axes[0], GT_SINGLE, "ground-truth parameters"), (axes[1], [30.0, 1.0], "too strong torque (α = 30)")]:
    lo2, lo1, hi1, hi2 = filter_bands(params)
    t = np.arange(len(lo1))
    ax.fill_between(t, lo2, hi2, color="r", alpha=0.15, lw=0, label="Posterior Predictive (95 % HDI)")
    ax.fill_between(t, lo1, hi1, color="r", alpha=0.3, lw=0, label="Posterior Predictive (68 % HDI)")
    ax.plot(p1[0, 1:], lw=1, label="Ground Truth")
    ax.set(xlabel="Time Step", ylabel="Orientation", title=title)
    ax.legend(fontsize=8)
savefig("fig26_phi_filter");

**Figure 26 (p. 72).** Red bands: 68 % and 95 % intervals of the posterior of the ABP orientation; blue: ground truth. With
the ground-truth parameters the filter follows the true dynamics closely; with a wrong (too strong) $f$ a significant gap opens,
which makes the data unlikely under the wrong parameters.

### Figure 27 (p. 74): Kalman filter vs. particle filter
Likelihood fields over amplitude $\alpha_1$ and wavelength parameter $k$ of a single-mode torque, from 100 type-A trajectories
of 250 steps, at three zoom levels (first two columns: log-likelihood; third column: linear density).
Coarse grid of `RESOLUTION` × `RESOLUTION` points per view (Figure 27: 1000 × 1000, computed on an HPC cluster);
particle filter with 2000 particles.

In [ ]:
FULL_RESOLUTION = True
RESOLUTION = 60
phi0_A = np.deg2rad(np.linspace(-45, 45, 100))
xA, yA, _ = wt.simulate_abp(250, A["dt"], phi0_A, A["Pe"], A["Dt"], A["Dr"], GT_SINGLE, rng, x0=-1.5, kind="single_mode")

VIEWS = [((1, 200), (0.1, 15)), ((1, 15), (0.1, 1.5)), ((1, 15), (0.1, 1.5))]

def in_blocks(loglik_fn, params, name, n_blocks=20):
    """Evaluate ``loglik_fn`` on the parameter rows in blocks; progress and ETA go to results/grid_progress.log."""
    out, t0 = [], time.time()
    blocks = np.array_split(params, n_blocks)
    with open(lt.utils.RESULTS_DIR / "grid_progress.log", "a") as log:
        for i, b in enumerate(blocks, 1):
            out.append(loglik_fn(b))
            eta = datetime.now() + timedelta(seconds=(time.time() - t0) / i * (n_blocks - i))
            log.write(f"{datetime.now():%Y-%m-%d %H:%M}  {name}  block {i}/{n_blocks}  ETA {eta:%H:%M}" + "\n"); log.flush()
    return np.concatenate(out)

def field(view, method):
    (a0, a1), (k0, k1) = view
    a, k = np.linspace(a0, a1, RESOLUTION), np.linspace(k0, k1, RESOLUTION)
    Ag, Kg = np.meshgrid(a, k)
    params = np.column_stack([Ag.ravel(), Kg.ravel()])
    if method == "kalman":
        ll = wt.loglik_kalman(xA, yA, phi0_A, A["dt"], A["Pe"], A["Dt"], A["Dr"], A["eps"], A["q2"], params)
    else:
        ll = in_blocks(lambda b: wt.loglik_particle_filter(xA, yA, phi0_A, A["dt"], A["Pe"], A["Dt"], A["Dr"], A["eps"],
                                                           wt.TORQUES["single_mode"], A["q2"], b, 2000, lt.SEED)[0],
                       params, f"fig27 view {view}")
    return {"logl": ll.reshape(Ag.shape), "a": a, "k": k}

fields = lt.cached(f"fig27_fields_res{RESOLUTION}_p2000", lambda: {
    f"{m}_{i}_{key}": val for m in ("kalman", "particle") for i, v in enumerate(VIEWS[:2]) for key, val in field(v, m).items()})

fig, axes = plt.subplots(2, 3, figsize=(12, 7.5))
for r, (m, label) in enumerate([("kalman", "Kalman Filter"), ("particle", "Particle Filter")]):
    for c in range(3):
        f = {key: fields[f"{m}_{min(c, 1)}_{key}"] for key in ("logl", "a", "k")}
        ll = f["logl"]
        img = ll if c < 2 else np.exp(ll - ll.max())
        ext = (f["a"][0], f["a"][-1], f["k"][0], f["k"][-1])
        ax = axes[r, c]
        ax.imshow(img if c < 2 else img, origin="lower", extent=ext, aspect="auto",
                  vmin=None if c == 2 else np.nanpercentile(img, 20))
        kk, aa = np.unravel_index(np.nanargmax(ll), ll.shape)
        ax.axhline(f["k"][kk], color="w", ls="--", lw=0.5); ax.axvline(f["a"][aa], color="w", ls="--", lw=0.5)
        ax.plot(*GT_SINGLE, "or", ms=5)
        if c == 0:
            ax.set_ylabel("k"); ax.text(0.03, 0.92, label, transform=ax.transAxes, color="k")
        if r == 1:
            ax.set_xlabel(r"$\alpha_1$")
axes[0, 0].plot([], [], "or", label="ground truth"); axes[0, 0].plot([], [], "w--", label="mode")
axes[0, 0].legend(loc="center right", fontsize=7, facecolor="0.7")
fig.tight_layout()
savefig("fig27_kalman_vs_particle");

**Figure 27 (p. 74).** Top row: linear Kalman filter; bottom row: particle filter. First two columns: log scale, third column:
linear. Red dot: ground truth; white dashed lines: mode of the distribution. The particle filter gives an unbiased estimate;
the Kalman filter recovers qualitative features but fails to recover the ground truth and shows fractal artefacts for low $k$
and high $\alpha_1$, attributed to its instability for large torques and strong oscillations.

### Posterior sampling
With more than a few parameters an exhaustive grid is no longer feasible, and we sample the posterior with the affine-invariant
ensemble sampler `emcee`. All walkers of an ensemble step are passed to the parallel particle filter at once
(`vectorize=True`); the prior is uniform within a box.

### B-spline representation
The spectral representation causes numerical issues beyond 3–4 modes: the posterior collapses into a thin, strongly
correlated distribution, since the Fourier modes are non-local and affected by all observations. Quadratic B-splines have
compact support,

$$B^{(2)}(x)=\sum_{i=0}^{N_B}\alpha_i\,\kappa\left((N-3)x+i-\tfrac12\right),\qquad
\kappa(u)=\begin{cases}\frac12u^2 & 0\le u\le1\\ \frac12(-2u^2+6u-3) & 1\le u\le2\\ \frac12(3-u)^2 & 2\le u\le3\\ 0&\text{else.}\end{cases}$$

In [ ]:
xs = np.linspace(-0.25, 1.25, 1000)
N = 10
plt.figure(figsize=(12, 3))
for i in range(N):
    coef = np.zeros(N); coef[i] = 1
    plt.plot(xs, [wt.bspline2(x, coef) for x in xs], label=f"basis function {i}" if i in (0, N - 1) else None)
plt.axvline(0, c="k", lw=1); plt.axvline(1, c="k", lw=1, label="interval [0, 1]")
plt.legend(fontsize=7, ncol=3); plt.grid(alpha=0.3)
savefig("fig29_bsplines");

**Figure 29 (p. 76).** Quadratic B-spline basis functions with equidistant knots for $x\in[0,1]$.

*Code:* `walltorque.torque_bspline(phi, coef)` continues the spline antisymmetrically and periodically in the orientation,
so that the torque vanishes for orientations along the wall normal.

### Validation on type-A data (Figure 30, p. 78)
20 type-A trajectories with a two-mode torque, $\alpha_1=\alpha_2=10$.

In [ ]:
GT_TWO = np.array([10.0, 10.0])
phi0_20 = np.deg2rad(np.linspace(-45, 45, 20))
x2, y2, _ = wt.simulate_abp(250, A["dt"], phi0_20, A["Pe"], A["Dt"], A["Dr"], GT_TWO, rng, x0=-1.5, kind="modes")

N_PARTICLES = 1500 if FULL_RESOLUTION else 100

def loglik(params, x, y, phi0, kind, n_particles=None, cfg=A):
    n_particles = N_PARTICLES if n_particles is None else n_particles
    return wt.loglik_particle_filter(x, y, phi0, cfg["dt"], cfg["Pe"], cfg["Dt"], cfg["Dr"], cfg["eps"],
                                     wt.TORQUES[kind], cfg["q2"], np.atleast_2d(params), n_particles, lt.SEED)[0]

def two_mode_grid(lim, res):
    a = np.linspace(*lim, res)
    A1, A2 = np.meshgrid(a, a)
    ll = in_blocks(lambda b: loglik(b, x2, y2, phi0_20, "modes", 1000), np.column_stack([A1.ravel(), A2.ravel()]),
                   f"fig30 grid {lim}")
    return {"a": a, "logl": ll.reshape(A1.shape)}

GRID30 = 40                       # coarse grid (Figure 30A/B: 2000 × 2000, computed on an HPC cluster)
g = lt.cached(f"fig30_two_mode_grid_res{GRID30}_p1000", lambda: {
    **{f"wide_{k}": v for k, v in two_mode_grid((1, 500), GRID30).items()},
    **{f"zoom_{k}": v for k, v in two_mode_grid((1, 20), GRID30).items()}})

In [ ]:
def run_emcee(log_prob, centre, half_width, n_walkers, n_steps, seed, init_spread=0.05, bounds=None):
    """Affine-invariant ensemble sampling with a uniform box prior centre ± half_width (or explicit ``bounds`` = (lo, hi));
    the walkers start in a small ball (init_spread x half_width) around the centre, where the posterior mass is expected.
    Returns the flat chain after discarding the first half as burn-in, and the acceptance fraction."""
    centre, hw = np.asarray(centre, float), np.broadcast_to(np.asarray(half_width, float), np.shape(centre))
    lo, hi = (centre - hw, centre + hw) if bounds is None else bounds
    def lp(theta):
        inside = np.all((theta >= lo) & (theta <= hi), axis=1)
        out = np.full(len(theta), -np.inf)
        if inside.any():
            out[inside] = log_prob(theta[inside])
        return out
    n_walkers = max(n_walkers, 2 * len(centre) + 2)      # the ensemble moves need more walkers than 2 x dimensions
    r = np.random.default_rng(seed)
    p0 = centre + init_spread * hw * r.standard_normal((n_walkers, len(centre)))
    sampler = emcee.EnsembleSampler(n_walkers, len(centre), lp, vectorize=True)
    sampler.random_state = np.random.RandomState(seed).get_state()       # reproducible proposals
    t0 = time.time()
    with open(lt.utils.RESULTS_DIR / "emcee_progress.log", "a") as log:
        log.write(f"{datetime.now():%Y-%m-%d %H:%M}  start: {len(centre)} parameters, {n_walkers} walkers, {n_steps} steps\n")
        for i, _ in enumerate(sampler.sample(p0, iterations=n_steps), 1):
            if i % 50 == 0 or i == n_steps:
                eta = datetime.now() + timedelta(seconds=(time.time() - t0) / i * (n_steps - i))
                log.write(f"{datetime.now():%Y-%m-%d %H:%M}  step {i}/{n_steps}  "
                          f"acceptance {np.mean(sampler.acceptance_fraction):.2f}  ETA {eta:%H:%M}\n"); log.flush()
    acc = np.mean(sampler.acceptance_fraction)
    tau = sampler.get_autocorr_time(tol=0)
    print(f"acceptance fraction {acc:.2f}, autocorrelation time (steps) {np.round(tau, 1)}, chain length {n_steps}")
    return sampler.get_chain(discard=n_steps // 2, flat=True), sampler.get_chain()

EMCEE_STEPS = 3000 if FULL_RESOLUTION else 1000
N_WALKERS = 32 if FULL_RESOLUTION else 16
N_KNOTS_A = 4
PHI_FIT = np.linspace(0, np.pi, 91)
centre_A = wt.bspline_projection([wt.torque_modes(p, GT_TWO) for p in PHI_FIT], PHI_FIT, N_KNOTS_A)

def chain_A_compute():
    flat, full = run_emcee(lambda th: loglik(th, x2, y2, phi0_20, "bspline"), centre_A, 15.0, N_WALKERS, EMCEE_STEPS, lt.SEED)
    return {"chain": flat, "walkers": full}

res_A = lt.cached(f"fig30_bspline_chain_{EMCEE_STEPS}_{N_PARTICLES}_{N_KNOTS_A}", chain_A_compute)
chain_A = res_A["chain"]

In [ ]:
ANG = np.linspace(0, np.pi, 181)          # inbound: 0-90 deg, outbound: 90-180 deg

def f_bands(chain, kind, ax, gt=None, n=2000, color="g"):
    """Density of the torque function f(phi) over posterior samples, with the 99 % interval."""
    idx = np.random.default_rng(lt.SEED).choice(len(chain), size=min(n, len(chain)), replace=False)
    fn = {"modes": wt.torque_modes, "bspline": wt.torque_bspline}[kind]
    F = np.array([[fn(a, th) for a in ANG] for th in chain[idx]])
    lo, hi = np.percentile(F, [0.5, 99.5], axis=0)
    ax.fill_between(np.rad2deg(ANG), lo, hi, color=color, alpha=0.25, label="posterior density of f (99 % interval)")
    ax.plot(np.rad2deg(ANG), lo, color=color, lw=1); ax.plot(np.rad2deg(ANG), hi, color=color, lw=1)
    ax.plot(np.rad2deg(ANG), np.median(F, axis=0), color=color, lw=2, ls=":", label="posterior median")
    if gt is not None:
        ax.plot(np.rad2deg(ANG), [wt.torque_modes(a, gt) for a in ANG], "r", lw=2, label="ground truth")
    ax.set(xlabel="angle of incidence φ − θ [deg]", ylabel="torque strength τσ", xticks=[0, 90, 180], xlim=(0, 180))
    ax.legend(fontsize=7)

fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))
for ax, key, log in [(axes[0], "wide", True), (axes[1], "zoom", False)]:
    a, ll = g[f"{key}_a"], g[f"{key}_logl"]
    img = ll if log else np.exp(ll - ll.max())
    ax.imshow(img, origin="lower", extent=(a[0], a[-1], a[0], a[-1]), aspect="auto",
              vmin=np.nanpercentile(img, 20) if log else None)
    j, i = np.unravel_index(np.nanargmax(ll), ll.shape)
    ax.axhline(a[j], color="w", ls="--", lw=0.7); ax.axvline(a[i], color="w", ls="--", lw=0.7)
    ax.plot(*GT_TWO, "or", label="ground truth"); ax.plot([], [], "w--", label="maximum")
    ax.set(xlabel=r"$\alpha_1$", ylabel=r"$\alpha_2$", title=("A (log density)" if log else "B (linear density)"))
    ax.legend(fontsize=7, facecolor="0.7")
f_bands(chain_A, "bspline", axes[2], gt=GT_TWO); axes[2].set_ylim(-50, 50)
axes[2].set_title(f"C (B-spline, {N_KNOTS_A} coefficients)")
fig.tight_layout()
savefig("fig30_typeA_test");

**Figure 30 (p. 78).** Bayesian inference from 20 type-A trajectories with known torque function ($\alpha_1=\alpha_2=10$, red).
A) Posterior structure with logarithmic density scaling, from a parameter sweep in the ground-truth parametrisation; dashed
lines: maximum. B) Zoom into the bulk of the posterior mass, linear density. C) Posterior of the torque function in a 4D
B-spline parametrisation. Most probability mass concentrates near the ground truth, with an anticorrelation of the two
amplitudes (modes partially interchangeable). The B-spline posterior captures $f$ for inbound orientations, while outbound
orientations are hardly informed — the dynamics never visit them inside the interaction region.

### Inference on rod data (Figure 31, p. 79)
20 type-R trajectories of rods with aspect ratio $p=1.5$ (2 test sites, Péclet number 100, $D_T=1$, $D_R$ from the
spherocylinder parametrisation, $\Delta t=10^{-4}$, 400 steps, start at $x=-2$, orientations in $\pm45°$); 1500 particles.

In [ ]:
P_ROD = 1.5
R = dict(Pe=100.0, Dt=1.0, Dr=wt.ratio_diffusion_rotational(P_ROD), dt=1e-4, eps=1.0, q2=1.0)
phi0_R = np.deg2rad(np.linspace(-45, 45, 20))
xR, yR, _ = wt.simulate_rod(400, R["dt"], phi0_R, R["Pe"], R["Dt"], P_ROD, rng, x0=-2.0)

The posterior mass is located first with a coarse two-mode grid (40 × 40, 50 particles). Both chains use a uniform prior
on [-10, 10] per parameter; the walkers start at the grid maximum (two modes) or at the B-spline representation of the
two-mode torque function at that maximum (B-splines). `N_KNOTS_R` sets the number of B-spline coefficients
(Figure 31 uses 10).

In [ ]:
def rod_grid():
    a = np.linspace(-10, 10, 40)
    A1, A2 = np.meshgrid(a, a)
    ll = loglik(np.column_stack([A1.ravel(), A2.ravel()]), xR, yR, phi0_R, "modes", 50, cfg=R).reshape(A1.shape)
    return {"a": a, "logl": ll}

grid_R = lt.cached("fig31_rod_grid", rod_grid)
j, i = np.unravel_index(np.argmax(grid_R["logl"]), grid_R["logl"].shape)
peak_R = np.array([grid_R["a"][i], grid_R["a"][j]])
N_KNOTS_R = 10
N_PARTICLES_R = 1500                      # filter particles for the rod chains
centre_R = wt.bspline_projection([wt.torque_modes(p, peak_R) for p in PHI_FIT], PHI_FIT, N_KNOTS_R)
print("two-mode maximum of the coarse grid:", peak_R)

def rod_chains():
    m_flat, m_full = run_emcee(lambda th: loglik(th, xR, yR, phi0_R, "modes", N_PARTICLES_R, cfg=R), peak_R, 10.0, N_WALKERS, EMCEE_STEPS,
                               lt.SEED, init_spread=0.02, bounds=(-10.0, 10.0))
    b_flat, b_full = run_emcee(lambda th: loglik(th, xR, yR, phi0_R, "bspline", N_PARTICLES_R, cfg=R), centre_R, 10.0, N_WALKERS, EMCEE_STEPS,
                               lt.SEED, init_spread=0.02, bounds=(-10.0, 10.0))
    return {"modes": m_flat, "modes_walkers": m_full, "bspline": b_flat, "bspline_walkers": b_full}

chains_R = lt.cached(f"fig31_rod_chains_{EMCEE_STEPS}_{N_PARTICLES_R}_{N_KNOTS_R}", rod_chains)

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5), sharey=True)
f_bands(chains_R["modes"], "modes", axes[0]); axes[0].set_ylim(-1, 1); axes[0].set_title("two modes")
f_bands(chains_R["bspline"], "bspline", axes[1]); axes[1].set_title(f"{N_KNOTS_R} B-spline coefficients")
fig.tight_layout()
savefig("fig31_rod_inference");

**Figure 31 (p. 79).** Posterior densities of the torque function for an ABP tracking an active rod's tip, from 20 rod
trajectories, parametrised by two modes (left) and B-spline coefficients (right); shaded: posterior density between the
99 % bounds. Most knowledge about $f$ is gathered for inbound orientations; the interval stays broad for parallel and outbound
orientations. With two modes the bandwidth is limited, correlating inbound and outbound directions; the B-splines lift this:
$f$ is not strongly defined for small angles of incidence (negative torques are preferred, without a dominant magnitude), and
in the outbound regime the posterior coincides with the prior.